# Section 06b Onboarding Notebook: Machine-Learned Interatomic Potentials — Training a MACE Model on MD Snapshots & Running MLIP Molecular Dynamics

### Overview & Pedagogical Design
*Ab initio* Born–Oppenheimer Molecular Dynamics (BOMD) requires solving the electronic Schrödinger equation at every nuclear step, restricting simulations to picoseconds and hundreds of atoms. **MACE (Multi-Atomic Cluster Expansion)** ([MACE Training Guide](https://mace-docs.readthedocs.io/en/latest/guide/training.html)) bridges quantum accuracy and classical MD speed by constructing higher-body-order equivariant message-passing features from local radial and angular atomic environments, predicting total potential energy $\hat{E}(\{\mathbf{R}_i\}) = \sum_i E_i^{(0)} + \sum_i \varepsilon_i(\{\mathbf{R}_{ij}\})$ and conservative analytical forces $\hat{\mathbf{F}}_i = -\nabla_{\mathbf{R}_i} \hat{E}$ at $\sim 10^4\times$ lower computational cost than DFT.

In this notebook, **no MACE data-processing, descriptor, training, or MLIP-MD functions are pre-solved for you**. You will implement **14 granular `# TODO` functions** covering the entire MACE workflow:
- **Part 1 — Generating Reference Ab-Initio MD Snapshots & Formatting `extxyz` Datasets (`TODO 1` – `TODO 4`)**
- **Part 2 — Local Atomic Descriptors: Smooth Cutoff Envelopes, Radial Bessel Basis & 3-Body Angular Features (`TODO 5` – `TODO 7`)**
- **Part 3 — Training a MACE-Style Energy+Force Potential & Evaluating Parity Metrics (`TODO 8` – `TODO 10`)**
- **Part 4 — Running MLIP-Driven Molecular Dynamics & Query-by-Committee Active Learning (`TODO 11` – `TODO 14`)**
- **Part 5 — Step-by-Step Checkpoint & Verification Suite (`run_section06b_checkpoints`)**

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
print("Section 06b environment initialized. Implement TODOs 1-14 and run `run_section06b_checkpoints()`!")

## Part 1: Reference Ab-Initio MD Snapshots, `IsolatedAtom` E0s & MACE `extxyz` Format (TODOs 1–4)

### Mathematical & Format Specifications
1. **Reference $\text{H}_2\text{O}$ Potential & Analytical Forces (`evaluate_water_reference_energy_and_forces` — TODO 1)**:
   For water with Oxygen at $\mathbf{R}_0$ and Hydrogens at $\mathbf{R}_1, \mathbf{R}_2$, bond vectors $\mathbf{v}_1 = \mathbf{R}_1 - \mathbf{R}_0$, $\mathbf{v}_2 = \mathbf{R}_2 - \mathbf{R}_0$, bond lengths $r_1 = \|\mathbf{v}_1\|, r_2 = \|\mathbf{v}_2\|$, unit vectors $\hat{\mathbf{u}}_1 = \mathbf{v}_1/r_1, \hat{\mathbf{u}}_2 = \mathbf{v}_2/r_2$, and angle $\theta = \arccos(\hat{\mathbf{u}}_1 \cdot \hat{\mathbf{u}}_2)$:
   $$E_{\text{ref}} = E_0^{\text{O}} + 2 E_0^{\text{H}} + \frac{1}{2}k_r\big[(r_1 - r_0)^2 + (r_2 - r_0)^2\big] - \frac{1}{3}k_3\big[(r_1 - r_0)^3 + (r_2 - r_0)^3\big] + \frac{1}{2}k_\theta(\theta - \theta_0)^2 + k_{rr}(r_1 - r_0)(r_2 - r_0)$$
   Evaluate $E_{\text{ref}}$ (in eV) and the exact analytical forces $\mathbf{F}_i = -\nabla_{\mathbf{R}_i} E_{\text{ref}}$ (in $\text{eV/\AA}$), verifying $\sum_{i=0}^2 \mathbf{F}_i = \mathbf{0}$ (translational invariance).

2. **Sampling Multi-Temperature MD Snapshots & `IsolatedAtom` E0 Frames (`sample_water_md_snapshots` — TODO 2)**:
   MACE subtracts the isolated-atom reference energies (`config_type=IsolatedAtom`, `E0s={1: -13.60, 8: -2041.80}`) so the neural network only learns the **atomization / binding energy** $\Delta E_{\text{bind}} = E_{\text{ref}} - \sum_i E_0^{(Z_i)}$ ([MACE Training Guide](https://mace-docs.readthedocs.io/en/latest/guide/training.html)).

3. **Writing & Parsing MACE Extended XYZ (`write_mace_extxyz` & `parse_mace_extxyz` — TODOs 3–4)**:
   Each frame in an `extxyz` file has:
   - Line 1: `<N_atoms>`
   - Line 2: `Properties=species:S:1:pos:R:3:REF_forces:R:3 REF_energy=<E_ev:.8f> config_type=<type> pbc="F F F"`
   - Lines $3 \dots N+2$: `<Symbol> <x> <y> <z> <Fx> <Fy> <Fz>`

In [ ]:
def evaluate_water_reference_energy_and_forces(
    positions: np.ndarray,
    e0_o: float = -2041.80,
    e0_h: float = -13.60,
    k_r: float = 36.5,
    k_3: float = 45.0,
    r0: float = 0.9572,
    k_theta: float = 4.35,
    theta0_deg: float = 104.52,
    k_rr: float = -0.65,
) -> tuple[float, np.ndarray]:
    '''
    TODO 1: Evaluate the anharmonic H2O reference energy (eV) and exact analytical forces (eV/Angstrom)
    for `positions` of shape (3, 3) [O, H1, H2]. Return (energy_ev, forces_ev_ang).
    '''
    # TODO 1: Compute bond lengths r1, r2, angle theta, total energy E, dE/dr1, dE/dr2, dE/dtheta,
    #         and apply the chain rule to compute forces on O, H1, and H2 (verifying sum(forces) == 0).
    raise NotImplementedError("Complete TODO 1: evaluate_water_reference_energy_and_forces")


def sample_water_md_snapshots(
    n_snapshots: int = 30, disp_scale_ang: float = 0.06, seed: int = 42, include_isolated_atoms: bool = True
) -> list[dict]:
    '''
    TODO 2: Generate a list of configuration dictionaries (each with keys 'symbols', 'positions',
    'energy_ev', 'forces_ev_ang', 'config_type'). If `include_isolated_atoms` is True, prepend
    two 1-atom configurations for isolated H (E = -13.60 eV) and isolated O (E = -2041.80 eV)
    with config_type="IsolatedAtom".
    '''
    # TODO 2: Prepend IsolatedAtom dicts if requested, then sample n_snapshots perturbed H2O geometries
    #         around equilibrium and evaluate their energy and forces via TODO 1.
    raise NotImplementedError("Complete TODO 2: sample_water_md_snapshots")


def write_mace_extxyz(configs: list[dict], filepath: str | Path) -> None:
    '''
    TODO 3: Write `configs` to `filepath` in MACE-compatible Extended XYZ (extxyz) format
    with `REF_energy` and `REF_forces` keys.
    '''
    # TODO 3: Format each configuration header and atom lines and write to filepath.
    raise NotImplementedError("Complete TODO 3: write_mace_extxyz")


def parse_mace_extxyz(filepath: str | Path) -> list[dict]:
    '''
    TODO 4: Parse a MACE `extxyz` file from `filepath` and return the list of configuration dicts
    with keys 'symbols', 'positions', 'energy_ev', 'forces_ev_ang', 'config_type'.
    '''
    # TODO 4: Read lines, parse atom count, extract REF_energy and config_type from the comment line,
    #         and parse species, positions (3 cols), and REF_forces (3 cols).
    raise NotImplementedError("Complete TODO 4: parse_mace_extxyz")

## Part 2: Local Atomic Descriptors — Smooth Cutoff Envelope, Radial Bessel Basis & 3-Body Angular Features (TODOs 5–7)

### Mathematical Specifications
MACE constructs rotationally invariant ($L=0$) and equivariant ($L>0$) atomic environment features from interatomic distances $r_{ij}$ and bond angles $\theta_{ijk}$ within a cutoff radius $r_{\text{cut}}$:
1. **Smooth Polynomial Cutoff Envelope (`smooth_cutoff_envelope` — TODO 5)**:
   $$f_{\text{cut}}(r) = \begin{cases} \frac{1}{2}\left[\cos\!\left(\frac{\pi r}{r_{\text{cut}}}\right) + 1\right] & r < r_{\text{cut}} \\ 0 & r \ge r_{\text{cut}} \end{cases}, \qquad \frac{df_{\text{cut}}}{dr} = \begin{cases} -\frac{\pi}{2 r_{\text{cut}}}\sin\!\left(\frac{\pi r}{r_{\text{cut}}}\right) & r < r_{\text{cut}} \\ 0 & r \ge r_{\text{cut}} \end{cases}$$

2. **Radial Bessel Basis Functions (`radial_bessel_basis` — TODO 6)**:
   For $n = 1, \dots, N_{\text{rad}}$:
   $$B_n(r) = \sqrt{\frac{2}{r_{\text{cut}}}}\,\frac{\sin\!\left(\frac{n\pi r}{r_{\text{cut}}}\right)}{r}\,f_{\text{cut}}(r)$$

3. **Three-Body Invariant Descriptor & Analytical Jacobian (`compute_water_feature_vector_and_jacobian` — TODO 7)**:
   To train a fast, exact force-matching MACE surrogate model $\hat{E}_{\text{bind}}(\mathbf{R}) = \mathbf{w}^\mathsf{T}\boldsymbol{\phi}(\mathbf{R})$ with analytical forces $\hat{\mathbf{F}}_i = -\left(\frac{\partial \boldsymbol{\phi}}{\partial \mathbf{R}_i}\right)^\mathsf{T}\mathbf{w}$, construct the 6-component invariant 2-body + 3-body feature vector $\boldsymbol{\phi}(\mathbf{R})$ of $\text{H}_2\text{O}$ from $(r_1, r_2, \theta)$:
   $$\boldsymbol{\phi}(\mathbf{R}) = \begin{bmatrix} 1,\; (r_1 - r_0)^2 + (r_2 - r_0)^2,\; (r_1 - r_0)^3 + (r_2 - r_0)^3,\; (\theta - \theta_0)^2,\; (r_1 - r_0)(r_2 - r_0),\; (r_1 + r_2 - 2r_0)(\theta - \theta_0) \end{bmatrix}^\mathsf{T}$$
   and its `(6, 3, 3)` analytical Cartesian gradient tensor $J_{m, i, \alpha} = \frac{\partial \phi_m}{\partial R_{i,\alpha}}$ so that $-\sum_m w_m J_{m, i, \alpha}$ yields exact conservative forces!

In [ ]:
def smooth_cutoff_envelope(r: np.ndarray, r_cut: float = 5.0) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 5: Evaluate the smooth cosine cutoff function f_cut(r) and its derivative df_cut/dr
    for array `r` with cutoff `r_cut`. Return (f_cut, df_cut_dr).
    '''
    # TODO 5: Use np.where(r < r_cut, ..., 0.0) to evaluate f_cut and df_cut_dr.
    raise NotImplementedError("Complete TODO 5: smooth_cutoff_envelope")


def radial_bessel_basis(r: np.ndarray, n_max: int = 6, r_cut: float = 5.0) -> np.ndarray:
    '''
    TODO 6: Evaluate the first `n_max` damped radial Bessel basis functions B_n(r) (n = 1..n_max)
    at distances `r` (1D array of length M). Return array of shape (M, n_max).
    '''
    # TODO 6: Call smooth_cutoff_envelope(r, r_cut) and evaluate sqrt(2/r_cut) * sin(n*pi*r/r_cut)/r * f_cut(r).
    raise NotImplementedError("Complete TODO 6: radial_bessel_basis")


def compute_water_feature_vector_and_jacobian(
    positions: np.ndarray, r0: float = 0.9572, theta0_deg: float = 104.52
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 7: Compute the 6-component invariant 2-body + 3-body feature vector phi(R) of shape (6,)
    and its analytical Cartesian Jacobian dphi_dR of shape (6, 3, 3) for water `positions` (3, 3).
    Return (phi, dphi_dR).
    '''
    # TODO 7: Compute r1, r2, theta and their Cartesian gradients dr1/dR, dr2/dR, dtheta/dR (each of shape (3, 3)),
    #         then assemble phi (shape 6) and dphi_dR (shape 6, 3, 3) via the chain rule.
    raise NotImplementedError("Complete TODO 7: compute_water_feature_vector_and_jacobian")

## Part 3: Training a MACE Model with Combined Energy + Force Loss & Parity Metrics (TODOs 8–10)

### Mathematical Specifications
In MACE training ([MACE Training Guide](https://mace-docs.readthedocs.io/en/latest/guide/training.html)), model parameters $\mathbf{w}$ are optimized by minimizing the weighted **Energy + Force loss**:
$$\mathcal{L}(\mathbf{w}) = w_E \frac{1}{N_{\text{frames}}}\sum_{n=1}^{N_{\text{frames}}} \left( \hat{E}_{\text{bind}}(\mathbf{R}^{(n)}; \mathbf{w}) - \Delta E_{\text{ref}}^{(n)} \right)^2 + w_F \frac{1}{3 N_{\text{atoms}} N_{\text{frames}}}\sum_{n=1}^{N_{\text{frames}}}\sum_{i=1}^{N_{\text{atoms}}} \left\| -\nabla_{\mathbf{R}_i}\hat{E}_{\text{bind}}(\mathbf{R}^{(n)}; \mathbf{w}) - \mathbf{F}_{\text{ref}, i}^{(n)} \right\|_2^2 + \lambda_{\text{reg}}\|\mathbf{w}\|_2^2$$
where $\Delta E_{\text{ref}}^{(n)} = E_{\text{ref}}^{(n)} - (E_0^{\text{O}} + 2 E_0^{\text{H}})$. Including force labels (`w_F >> w_E`, e.g., `--forces_weight=100.0`) provides $3N_{\text{atoms}}$ gradient constraints per MD frame, dramatically improving data efficiency!

In [ ]:
def extract_isolated_atom_e0s(configs: list[dict]) -> dict[str, float]:
    '''
    TODO 8: Scan `configs` for frames with config_type == "IsolatedAtom" and return a dictionary
    mapping element symbol (e.g. "O", "H") to its isolated atomic energy E0 (eV).
    '''
    # TODO 8: Extract {cfg["symbols"][0]: float(cfg["energy_ev"])} for all IsolatedAtom frames.
    raise NotImplementedError("Complete TODO 8: extract_isolated_atom_e0s")


def fit_mace_surrogate_potential(
    train_configs: list[dict],
    e0_map: dict[str, float],
    energy_weight: float = 1.0,
    forces_weight: float = 100.0,
    l2_reg: float = 1e-8,
) -> np.ndarray:
    '''
    TODO 9: Fit the 6-parameter MACE surrogate weight vector `w` by minimizing the weighted
    Energy + Force least-squares loss over all non-IsolatedAtom frames in `train_configs`.
    Return `w` of shape (6,).
    '''
    # TODO 9: For each MD frame, compute binding energy target dE = E_ref - sum(E0), feature vector phi,
    #         and force Jacobian -dphi_dR. Stack weighted design matrix rows and solve via Ridge / lstsq.
    raise NotImplementedError("Complete TODO 9: fit_mace_surrogate_potential")


def predict_and_evaluate_parity_metrics(
    configs: list[dict], w: np.ndarray, e0_map: dict[str, float]
) -> dict[str, float | np.ndarray]:
    '''
    TODO 10: Predict total energies (eV) and forces (eV/Ang) for all MD frames in `configs`
    using trained weights `w` and `e0_map`, and return a dict with keys:
    'energy_mae_mev', 'energy_rmse_mev', 'force_mae_mev_ang', 'force_rmse_mev_ang',
    'pred_energies', 'ref_energies', 'pred_forces', 'ref_forces'.
    '''
    # TODO 10: Evaluate E_pred = sum(E0) + phi @ w and F_pred = -einsum("m,mia->ia", w, dphi_dR)
    #          across all MD frames and compute MAE and RMSE in meV and meV/Angstrom.
    raise NotImplementedError("Complete TODO 10: predict_and_evaluate_parity_metrics")

## Part 4: Running MACE-Driven Molecular Dynamics & Committee Active Learning (TODOs 11–14)

### Mathematical Specifications
1. **Single-Frame MACE Energy & Force Predictor (`predict_mace_energy_and_forces` — TODO 11)**:
   Given nuclear coordinates $\mathbf{R}$, evaluate $\hat{E}(\mathbf{R}) = E_0^{\text{sum}} + \boldsymbol{\phi}(\mathbf{R})^\mathsf{T}\mathbf{w}$ and $\hat{\mathbf{F}}(\mathbf{R}) = -\nabla_{\mathbf{R}}\boldsymbol{\phi}(\mathbf{R})^\mathsf{T}\mathbf{w}$.
2. **MACE-Driven 3D Velocity Verlet MD (`run_mace_md_trajectory` — TODO 12)**:
   Propagate the 3-atom $\text{H}_2\text{O}$ molecule in 3D Cartesian space using the trained MACE surrogate potential (`dt_fs` in femtoseconds, converting $\text{eV/\AA}$ and $\text{amu}$ consistently).
3. **Committee Force Ensemble & Active-Learning Uncertainty (`train_mace_committee_and_uncertainty` & `select_active_learning_frames` — TODOs 13–14)**:
   Train $M_{\text{comm}} = 4$ models on bootstrap-resampled subsets of the training snapshots. At any new MD configuration $\mathbf{R}$, evaluate the committee forces $\{\hat{\mathbf{F}}^{(m)}(\mathbf{R})\}_{m=1}^M$, mean force $\bar{\mathbf{F}}_i = \frac{1}{M}\sum_{m=1}^M \hat{\mathbf{F}}_i^{(m)}$, and per-atom epistemic force uncertainty ([MACE Active Learning](https://mace-docs.readthedocs.io/en/latest/guide/training.html)):
   $$\sigma_{F, i}(\mathbf{R}) = \sqrt{\frac{1}{M}\sum_{m=1}^M \left\| \hat{\mathbf{F}}_i^{(m)}(\mathbf{R}) - \bar{\mathbf{F}}_i(\mathbf{R}) \right\|_2^2}, \qquad \sigma_F^{\max}(\mathbf{R}) = \max_{i} \sigma_{F, i}(\mathbf{R})$$
   Flag frames where $\sigma_F^{\max} \in [\tau_{\text{min}}, \tau_{\text{max}}]$ for *ab initio* DFT relabeling!

In [ ]:
def predict_mace_energy_and_forces(
    positions: np.ndarray, w: np.ndarray, e0_sum: float = -2069.00
) -> tuple[float, np.ndarray]:
    '''
    TODO 11: Predict total energy (eV) and (3, 3) Cartesian forces (eV/Ang) for a single H2O geometry.
    '''
    # TODO 11: Call compute_water_feature_vector_and_jacobian(positions) and contract with `w`.
    raise NotImplementedError("Complete TODO 11: predict_mace_energy_and_forces")


def run_mace_md_trajectory(
    positions_init: np.ndarray,
    velocities_init: np.ndarray,
    masses_amu: np.ndarray,
    w: np.ndarray,
    dt_fs: float = 0.5,
    n_steps: int = 200,
) -> dict[str, np.ndarray | float]:
    '''
    TODO 12: Propagate a 3D NVE molecular dynamics trajectory for H2O using `predict_mace_energy_and_forces`
    and Velocity Verlet. (Use conversion factor EV_ANG_AMU_TO_ANG_FS2 = 0.00964853321 for a = F / m in Ang/fs^2,
    and KE_ev = 0.5 * sum_i m_i * ||v_i||^2 / 0.00964853321).
    Return a dict with keys 'times_fs', 'positions_traj', 'kinetic_ev', 'potential_ev', 'total_ev', 'max_drift_ev'.
    '''
    # TODO 12: Integrate 3D Velocity Verlet for n_steps and record energies and positions.
    raise NotImplementedError("Complete TODO 12: run_mace_md_trajectory")


def compute_committee_force_uncertainty(
    committee_forces: np.ndarray,
) -> tuple[np.ndarray, float]:
    '''
    TODO 13: Given `committee_forces` of shape (M_models, N_atoms, 3), compute the per-atom
    RMS force standard deviation sigma_{F,i} of shape (N_atoms,) and max_sigma_f = max(sigma_{F,i}).
    Return (atom_sigma_f, max_sigma_f).
    '''
    # TODO 13: Compute mean_f = mean(committee_forces, axis=0), diff_sq = sum((committee_forces - mean_f)^2, axis=-1),
    #          atom_sigma_f = sqrt(mean(diff_sq, axis=0)), and return (atom_sigma_f, float(max(atom_sigma_f))).
    raise NotImplementedError("Complete TODO 13: compute_committee_force_uncertainty")


def select_active_learning_frames(
    candidate_positions: list[np.ndarray],
    committee_weights: list[np.ndarray],
    sigma_threshold: float = 0.05,
    sigma_max_cap: float = 5.0,
) -> dict[str, list[int] | np.ndarray]:
    '''
    TODO 14: Evaluate committee force uncertainty across `candidate_positions` using `committee_weights`
    and return a dict with keys 'max_uncertainties' (1D array) and 'selected_frame_indices'
    (indices where sigma_threshold <= max_sigma_f <= sigma_max_cap).
    '''
    # TODO 14: For each geometry, evaluate forces across all models in committee_weights,
    #          call compute_committee_force_uncertainty, and filter indices in [sigma_threshold, sigma_max_cap].
    raise NotImplementedError("Complete TODO 14: select_active_learning_frames")

## Part 5: Step-by-Step Checkpoint & Verification Suite

Run `run_section06b_checkpoints()` below as you implement each `# TODO` function to test your MACE dataset pipeline, descriptors, force-matching regression, MLIP MD integrator, and committee active-learning selector!

In [2]:
def run_section06b_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 14 #TODO functions in Section 06b (MACE MLIP & MD).'''
    passed = 0
    total_todos = 14

    try:
        r_eq_h2o = np.array([[0.0, 0.0, 0.0], [0.0, -0.757, 0.587], [0.0, 0.757, 0.587]])
        e_eq, f_eq = evaluate_water_reference_energy_and_forces(r_eq_h2o)
        assert np.linalg.norm(np.sum(f_eq, axis=0)) < 1e-12 and np.linalg.norm(f_eq) < 0.05
        # Check finite-difference force accuracy on a distorted geometry
        r_dist = r_eq_h2o + np.array([[0.01, -0.02, 0.01], [-0.03, 0.04, -0.02], [0.02, -0.02, 0.01]])
        _, f_an = evaluate_water_reference_energy_and_forces(r_dist)
        eps_fd = 1e-6
        r_p = r_dist.copy(); r_p[1, 2] += eps_fd
        r_m = r_dist.copy(); r_m[1, 2] -= eps_fd
        f_fd_1z = -(evaluate_water_reference_energy_and_forces(r_p)[0] - evaluate_water_reference_energy_and_forces(r_m)[0]) / (2 * eps_fd)
        assert np.isclose(f_an[1, 2], f_fd_1z, atol=1e-6)
        passed += 1
        print("[PASS] TODO 1 : evaluate_water_reference_energy_and_forces (Analytical forces match FD to < 1e-6 eV/A)")

        snaps = sample_water_md_snapshots(n_snapshots=24, seed=42, include_isolated_atoms=True)
        assert len(snaps) == 26 and snaps[0]["config_type"] == "IsolatedAtom"
        passed += 1
        print("[PASS] TODO 2 : sample_water_md_snapshots")

        tmp_xyz = Path("/tmp/test_mace_water.xyz")
        write_mace_extxyz(snaps, tmp_xyz)
        assert tmp_xyz.exists() and "REF_energy=" in tmp_xyz.read_text()
        passed += 1
        print("[PASS] TODO 3 : write_mace_extxyz")

        parsed = parse_mace_extxyz(tmp_xyz)
        assert len(parsed) == 26 and np.allclose(parsed[5]["positions"], snaps[5]["positions"], atol=1e-5)
        passed += 1
        print("[PASS] TODO 4 : parse_mace_extxyz")

        fc, dfc = smooth_cutoff_envelope(np.array([0.0, 2.5, 5.0, 6.0]), r_cut=5.0)
        assert np.isclose(fc[0], 1.0) and np.isclose(fc[1], 0.5) and fc[2] == 0.0 and dfc[2] == 0.0
        passed += 1
        print("[PASS] TODO 5 : smooth_cutoff_envelope")

        b_mat = radial_bessel_basis(np.array([1.0, 2.0, 3.0]), n_max=6, r_cut=5.0)
        assert b_mat.shape == (3, 6)
        passed += 1
        print("[PASS] TODO 6 : radial_bessel_basis")

        phi_d, jac_d = compute_water_feature_vector_and_jacobian(r_dist)
        assert phi_d.shape == (6,) and jac_d.shape == (6, 3, 3)
        # Verify feature Jacobian against finite differences
        phi_p, _ = compute_water_feature_vector_and_jacobian(r_p)
        phi_m, _ = compute_water_feature_vector_and_jacobian(r_m)
        assert np.allclose(jac_d[:, 1, 2], (phi_p - phi_m) / (2 * eps_fd), atol=1e-6)
        passed += 1
        print("[PASS] TODO 7 : compute_water_feature_vector_and_jacobian")

        e0_map = extract_isolated_atom_e0s(snaps)
        assert np.isclose(e0_map["O"], -2041.80) and np.isclose(e0_map["H"], -13.60)
        passed += 1
        print("[PASS] TODO 8 : extract_isolated_atom_e0s")

        w_fit = fit_mace_surrogate_potential(snaps[:18], e0_map, energy_weight=1.0, forces_weight=100.0)
        assert w_fit.shape == (6,)
        passed += 1
        print("[PASS] TODO 9 : fit_mace_surrogate_potential")

        metrics = predict_and_evaluate_parity_metrics(snaps[18:], w_fit, e0_map)
        assert metrics["energy_rmse_mev"] < 1.0 and metrics["force_rmse_mev_ang"] < 1.0
        passed += 1
        print(f"[PASS] TODO 10: predict_and_evaluate_parity_metrics (Test RMSE_E = {metrics['energy_rmse_mev']:.4f} meV, RMSE_F = {metrics['force_rmse_mev_ang']:.4f} meV/A)")

        e_p1, f_p1 = predict_mace_energy_and_forces(r_dist, w_fit)
        assert np.isclose(e_p1, evaluate_water_reference_energy_and_forces(r_dist)[0], atol=1e-4)
        passed += 1
        print("[PASS] TODO 11: predict_mace_energy_and_forces")

        md_res = run_mace_md_trajectory(
            r_dist, np.zeros((3, 3)), np.array([15.999, 1.008, 1.008]), w_fit, dt_fs=0.25, n_steps=100
        )
        assert md_res["max_drift_ev"] < 1e-4
        passed += 1
        print(f"[PASS] TODO 12: run_mace_md_trajectory (Max NVE energy drift = {md_res['max_drift_ev']:.2e} eV)")

        cf_mock = np.array([f_p1, f_p1 + 0.04, f_p1 - 0.04, f_p1])
        atom_sig, max_sig = compute_committee_force_uncertainty(cf_mock)
        assert atom_sig.shape == (3,) and max_sig > 0.01
        passed += 1
        print(f"[PASS] TODO 13: compute_committee_force_uncertainty (max_sigma_F = {max_sig:.4f} eV/A)")

        w_comm = [w_fit, w_fit * np.array([1.0, 1.02, 0.95, 1.01, 1.0, 0.98]), w_fit * np.array([1.0, 0.98, 1.05, 0.99, 1.0, 1.02])]
        al_res = select_active_learning_frames([r_eq_h2o, r_dist * 1.15], w_comm, sigma_threshold=0.01)
        assert 1 in al_res["selected_frame_indices"] and 0 not in al_res["selected_frame_indices"]
        passed += 1
        print("[PASS] TODO 14: select_active_learning_frames (High-strain geometry flagged for DFT relabeling!)")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 06b (MACE MLIP & MD) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section06b_checkpoints()
print("Section 06b checkpoint suite loaded. Implement TODOs 1-14 and call `run_section06b_checkpoints()`!")

Section 06b checkpoint suite loaded. Implement TODOs 1-14 and call `run_section06b_checkpoints()`!
